In [1]:
!mkdir phishing_analyzer

In [2]:
%cd phishing_analyzer

/content/phishing_analyzer


In [4]:
import requests
print(requests.__version__)

2.34.2


In [5]:
!pip install -q \
pandas \
numpy \
scikit-learn \
joblib \
tqdm \
beautifulsoup4 \
plotly \
langchain \
langchain-community \
openai


In [ ]:
import os

os.environ["OPENROUTER_API_KEY"] = ""

In [7]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

response = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role":"user",
            "content":"Reply only with OK"
        }
    ]
)

print(response.choices[0].message.content)

OK


In [8]:
files = [
    "parser.py",
    "feature_extractor.py",
    "classifier.py",
    "rule_engine.py",
    "risk_engine.py",
    "report_generator.py",
    "dashboard.py",
    "app.py"
]

for f in files:
    open(f, "w").close()

In [9]:
!wget https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2

--2026-05-30 14:55:02--  https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2
Resolving spamassassin.apache.org (spamassassin.apache.org)... 151.101.2.132, 2a04:4e42::644
Connecting to spamassassin.apache.org (spamassassin.apache.org)|151.101.2.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1612216 (1.5M) [application/x-bzip2]
Saving to: ‘20030228_easy_ham.tar.bz2’

20030228_easy_ham.t 100%[===================>]   1.54M  --.-KB/s    in 0.04s   

2026-05-30 14:55:03 (35.8 MB/s) - ‘20030228_easy_ham.tar.bz2’ saved [1612216/1612216]



In [10]:
!wget https://spamassassin.apache.org/old/publiccorpus/20030228_spam.tar.bz2

--2026-05-30 14:55:14--  https://spamassassin.apache.org/old/publiccorpus/20030228_spam.tar.bz2
Resolving spamassassin.apache.org (spamassassin.apache.org)... 151.101.2.132, 2a04:4e42::644
Connecting to spamassassin.apache.org (spamassassin.apache.org)|151.101.2.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1183768 (1.1M) [application/x-bzip2]
Saving to: ‘20030228_spam.tar.bz2’

20030228_spam.tar.b 100%[===================>]   1.13M  --.-KB/s    in 0.04s   

2026-05-30 14:55:15 (29.3 MB/s) - ‘20030228_spam.tar.bz2’ saved [1183768/1183768]



In [11]:
!tar -xjf 20030228_easy_ham.tar.bz2
!tar -xjf 20030228_spam.tar.bz2

In [12]:
import os

def read_email(file_path):
    try:
        with open(file_path, "r", encoding="latin-1") as f:
            return f.read()
    except:
        return ""

In [13]:
import pandas as pd

emails = []
labels = []

# Ham emails
for file in os.listdir("easy_ham"):
    path = os.path.join("easy_ham", file)

    if os.path.isfile(path):
        emails.append(read_email(path))
        labels.append(0)

# Spam emails
for file in os.listdir("spam"):
    path = os.path.join("spam", file)

    if os.path.isfile(path):
        emails.append(read_email(path))
        labels.append(1)

df = pd.DataFrame({
    "email": emails,
    "label": labels
})

print(df.shape)
df.head()

(3002, 2)


,email,label
0,From ilug-admin@linux.ie Mon Sep 2 13:14:35 ...,0
1,From ilug-admin@linux.ie Thu Aug 29 16:52:52 ...,0
2,From rssfeeds@jmason.org Tue Oct 1 10:37:14 ...,0
3,From rpm-list-admin@freshrpms.net Tue Oct 8 ...,0
4,Return-Path: tim.one@comcast.net\nDelivery-Dat...,0


In [14]:
import re

def preprocess(text):

    text = text.lower()

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_email"] = df["email"].apply(preprocess)

In [15]:
print(df["label"].value_counts())

label
0    2501
1     501
Name: count, dtype: int64


In [16]:
df.to_csv("spamassassin_processed.csv", index=False)

In [17]:
from sklearn.model_selection import train_test_split

X = df["clean_email"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(2401,)
(601,)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(2401, 10000)


In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(
    X_train_tfidf,
    y_train
)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [20]:
from sklearn.metrics import classification_report

preds = model.predict(X_test_tfidf)

print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

           0       1.00      0.97      0.99       501
           1       0.88      0.99      0.93       100

    accuracy                           0.98       601
   macro avg       0.94      0.98      0.96       601
weighted avg       0.98      0.98      0.98       601



In [21]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(
    y_test,
    preds
)

print("Accuracy:", acc)

Accuracy: 0.9767054908485857


In [22]:
sample = """
URGENT:
Your account has been suspended.

Click immediately:

http://secure-login-update.xyz
"""

sample = preprocess(sample)

sample_vector = vectorizer.transform([sample])

prob = model.predict_proba(sample_vector)[0][1]

print(prob)

0.7712964785788154


In [23]:
import os

os.makedirs(
    "models",
    exist_ok=True
)

In [24]:
# parser.py

import re

URL_REGEX = r'https?://[^\s]+'

def extract_urls(text):
    return re.findall(URL_REGEX, text)

def parse_email_text(email_text):

    urls = extract_urls(email_text)

    return {
        "body": email_text,
        "urls": urls
    }

In [25]:
sample = """
URGENT

Verify account immediately

https://secure-login.xyz
"""

parsed = parse_email_text(sample)

print(parsed)

{'body': '\nURGENT\n\nVerify account immediately\n\nhttps://secure-login.xyz\n', 'urls': ['https://secure-login.xyz']}


In [26]:
# feature_extractor.py

import re

In [27]:
URGENT_WORDS = [
    "urgent",
    "verify",
    "account suspended",
    "click now",
    "immediate action",
    "password reset",
    "security alert"
]

In [28]:
def detect_urgent_words(text):

    text = text.lower()

    found = []

    for word in URGENT_WORDS:

        if word in text:
            found.append(word)

    return found

In [29]:
def capital_ratio(text):

    letters = [c for c in text if c.isalpha()]

    if len(letters) == 0:
        return 0

    capitals = [c for c in letters if c.isupper()]

    return len(capitals) / len(letters)

In [30]:
def is_ip_url(url):

    pattern = r'https?://\d+\.\d+\.\d+\.\d+'

    return bool(
        re.search(pattern, url)
    )

In [31]:
SHORTENERS = [
    "bit.ly",
    "tinyurl.com",
    "goo.gl",
    "t.co"
]

def is_shortened_url(url):

    return any(
        s in url
        for s in SHORTENERS
    )

In [32]:
def extract_features(parsed_email):

    urls = parsed_email["urls"]

    body = parsed_email["body"]

    features = {}

    features["url_count"] = len(urls)

    features["urgent_words"] = detect_urgent_words(body)

    features["urgent_count"] = len(
        features["urgent_words"]
    )

    features["capital_ratio"] = capital_ratio(body)

    features["ip_urls"] = sum(
        is_ip_url(url)
        for url in urls
    )

    features["short_urls"] = sum(
        is_shortened_url(url)
        for url in urls
    )

    return features

In [34]:
!cat parser.py

In [35]:
import re

URL_REGEX = r'https?://[^\s]+'

def extract_urls(text):
    return re.findall(URL_REGEX, text)

def parse_email_text(email_text):

    urls = extract_urls(email_text)

    return {
        "body": email_text,
        "urls": urls
    }

In [36]:
import re

URGENT_WORDS = [
    "urgent",
    "verify",
    "account suspended",
    "click now",
    "immediate action",
    "password reset",
    "security alert"
]

SHORTENERS = [
    "bit.ly",
    "tinyurl.com",
    "goo.gl",
    "t.co"
]

def detect_urgent_words(text):

    text = text.lower()

    found = []

    for word in URGENT_WORDS:
        if word in text:
            found.append(word)

    return found


def capital_ratio(text):

    letters = [c for c in text if c.isalpha()]

    if len(letters) == 0:
        return 0

    capitals = [c for c in letters if c.isupper()]

    return len(capitals) / len(letters)


def is_ip_url(url):

    pattern = r'https?://\d+\.\d+\.\d+\.\d+'

    return bool(re.search(pattern, url))


def is_shortened_url(url):

    return any(
        s in url
        for s in SHORTENERS
    )


def extract_features(parsed_email):

    urls = parsed_email["urls"]

    body = parsed_email["body"]

    return {
        "url_count": len(urls),
        "urgent_words": detect_urgent_words(body),
        "urgent_count": len(
            detect_urgent_words(body)
        ),
        "capital_ratio": capital_ratio(body),
        "ip_urls": sum(
            is_ip_url(url)
            for url in urls
        ),
        "short_urls": sum(
            is_shortened_url(url)
            for url in urls
        )
    }

In [37]:
sample = """
URGENT

Your account has been suspended.

Verify immediately.

http://192.168.1.1/login

https://bit.ly/reset123
"""

parsed = parse_email_text(sample)

features = extract_features(parsed)

print(features)

{'url_count': 2, 'urgent_words': ['urgent', 'verify'], 'urgent_count': 2, 'capital_ratio': 0.10810810810810811, 'ip_urls': 1, 'short_urls': 1}


In [38]:
def calculate_rule_score(features):

    score = 0

    indicators = []

    # Urgency
    if features["urgent_count"] > 0:
        score += 10
        indicators.append(
            f"Urgent language detected ({features['urgent_words']})"
        )

    # Too many URLs
    if features["url_count"] > 3:
        score += 10
        indicators.append(
            "Multiple URLs detected"
        )

    # IP URL
    if features["ip_urls"] > 0:
        score += 15
        indicators.append(
            "IP-based URL detected"
        )

    # URL shortener
    if features["short_urls"] > 0:
        score += 10
        indicators.append(
            "Shortened URL detected"
        )

    # Excessive capitalization
    if features["capital_ratio"] > 0.30:
        score += 5
        indicators.append(
            "Excessive capitalization"
        )

    score = min(score, 100)

    return {
        "rule_score": score,
        "indicators": indicators
    }

In [39]:
rule_result = calculate_rule_score(features)

print(rule_result)

{'rule_score': 35, 'indicators': ["Urgent language detected (['urgent', 'verify'])", 'IP-based URL detected', 'Shortened URL detected']}


In [41]:
import os

print(os.getcwd())

/content/phishing_analyzer


In [42]:
!ls

20030228_easy_ham.tar.bz2  easy_ham		 report_generator.py
20030228_spam.tar.bz2	   feature_extractor.py  risk_engine.py
app.py			   models		 rule_engine.py
classifier.py		   parser.py		 spam
dashboard.py		   __pycache__		 spamassassin_processed.csv


In [43]:
import joblib

joblib.dump(
    model,
    "models/model.pkl"
)

joblib.dump(
    vectorizer,
    "models/vectorizer.pkl"
)

['models/vectorizer.pkl']

In [44]:
import joblib

model = joblib.load(
    "models/model.pkl"
)

vectorizer = joblib.load(
    "models/vectorizer.pkl"
)

In [45]:
print(type(model))
print(type(vectorizer))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [46]:
def get_ml_score(email_text):

    clean_text = preprocess(email_text)

    vector = vectorizer.transform(
        [clean_text]
    )

    probability = model.predict_proba(
        vector
    )[0][1]

    return probability * 100

In [47]:
ml_score = get_ml_score(sample)

print(ml_score)

66.08640459462659


In [48]:
def calculate_final_risk(
    ml_score,
    rule_score
):

    final_score = (
        0.7 * ml_score
        +
        0.3 * rule_score
    )

    return round(final_score, 2)

In [49]:
def classify_risk(score):

    if score < 25:
        return "Low"

    elif score < 50:
        return "Medium"

    elif score < 75:
        return "High"

    else:
        return "Critical"

In [50]:
rule_score = 35

ml_score = 66.09

final_score = calculate_final_risk(
    ml_score,
    rule_score
)

risk_level = classify_risk(
    final_score
)

print(final_score)
print(risk_level)

56.76
High


In [51]:
def analyze_email(email_text):

    # Parse
    parsed = parse_email_text(
        email_text
    )

    # Features
    features = extract_features(
        parsed
    )

    # Rules
    rule_result = calculate_rule_score(
        features
    )

    # ML
    ml_score = get_ml_score(
        email_text
    )

    # Fusion
    final_score = calculate_final_risk(
        ml_score,
        rule_result["rule_score"]
    )

    risk_level = classify_risk(
        final_score
    )

    return {
        "ml_score": round(
            ml_score,
            2
        ),
        "rule_score": rule_result[
            "rule_score"
        ],
        "final_score": final_score,
        "risk_level": risk_level,
        "indicators": rule_result[
            "indicators"
        ]
    }

In [52]:
email = """
URGENT

Your Microsoft account has been suspended.

Verify immediately.

Click below:

http://192.168.1.1/login

https://bit.ly/reset123
"""

result = analyze_email(email)

print(result)

{'ml_score': np.float64(77.0), 'rule_score': 35, 'final_score': np.float64(64.4), 'risk_level': 'High', 'indicators': ["Urgent language detected (['urgent', 'verify'])", 'IP-based URL detected', 'Shortened URL detected']}


In [53]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

In [54]:
response = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role":"user",
            "content":"Reply only with OK"
        }
    ]
)

print(response.choices[0].message.content)

OK


In [55]:
def build_prompt(
    email_text,
    analysis_result
):

    return f"""
You are a cybersecurity phishing analyst.

Analyze the following email.

EMAIL:
{email_text}

RISK SCORE:
{analysis_result['final_score']}

RISK LEVEL:
{analysis_result['risk_level']}

INDICATORS:
{analysis_result['indicators']}

Provide:

1. Threat Summary

2. Risk Explanation

3. Attack Category
   (Credential Theft, Malware, Business Email Compromise, Scam, Other)

4. Recommendations

5. Security Awareness Tips

Keep the response concise and professional.
"""

In [56]:
def generate_threat_report(
    email_text,
    analysis_result
):

    prompt = build_prompt(
        email_text,
        analysis_result
    )

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [57]:
report = generate_threat_report(
    email,
    result
)

print(report)

### 1. Threat Summary
The email is a phishing attempt designed to deceive the recipient into providing their Microsoft account credentials. It employs urgent language to create a sense of panic and urgency, prompting immediate action. The inclusion of both an IP-based URL and a shortened URL increases the likelihood of malicious intent.

### 2. Risk Explanation
The risk score of 64.4 indicates a high potential for harm. The use of urgent language ("URGENT," "Verify immediately") is a common tactic in phishing attacks to elicit quick responses without careful consideration. The IP-based URL (http://192.168.1.1/login) is suspicious as it typically refers to local network addresses, while the shortened URL (https://bit.ly/reset123) obscures the final destination, making it difficult for users to assess its safety.

### 3. Attack Category
Credential Theft

### 4. Recommendations
- **Do Not Click Links**: Advise users to avoid clicking on any links in the email.
- **Verify Account Status**:

In [58]:
DANGEROUS_EXTENSIONS = [
    ".exe",
    ".bat",
    ".js",
    ".vbs",
    ".scr",
    ".ps1",
    ".docm",
    ".xlsm"
]

In [59]:
def analyze_attachments(files):

    findings = []

    score = 0

    for file in files:

        lower = file.lower()

        for ext in DANGEROUS_EXTENSIONS:

            if lower.endswith(ext):

                findings.append(
                    f"Dangerous attachment: {file}"
                )

                score += 20

    return {
        "attachment_score": score,
        "findings": findings
    }

In [60]:
attachments = [
    "invoice.pdf.exe",
    "salary.xlsx",
    "resume.docm"
]

print(
    analyze_attachments(
        attachments
    )
)

{'attachment_score': 40, 'findings': ['Dangerous attachment: invoice.pdf.exe', 'Dangerous attachment: resume.docm']}


In [61]:
def analyze_headers(
    from_email,
    reply_to
):

    findings = []

    score = 0

    if (
        reply_to
        and
        from_email.lower()
        != reply_to.lower()
    ):

        findings.append(
            "Reply-To mismatch detected"
        )

        score += 15

    return {
        "header_score": score,
        "findings": findings
    }

In [62]:
print(
    analyze_headers(
        "support@microsoft.com",
        "random123@gmail.com"
    )
)

{'header_score': 15, 'findings': ['Reply-To mismatch detected']}


In [63]:
def analyze_email_complete(
    email_text,
    attachments=None,
    from_email=None,
    reply_to=None
):

    if attachments is None:
        attachments = []

    # Existing analysis
    parsed = parse_email_text(email_text)

    features = extract_features(parsed)

    rule_result = calculate_rule_score(features)

    ml_score = get_ml_score(email_text)

    # Attachment analysis
    attachment_result = analyze_attachments(
        attachments
    )

    # Header analysis
    header_result = analyze_headers(
        from_email,
        reply_to
    )

    # Enhanced rule score
    total_rule_score = (
        rule_result["rule_score"]
        +
        attachment_result["attachment_score"]
        +
        header_result["header_score"]
    )

    total_rule_score = min(
        total_rule_score,
        100
    )

    # Final risk
    final_score = calculate_final_risk(
        ml_score,
        total_rule_score
    )

    risk_level = classify_risk(
        final_score
    )

    indicators = (
        rule_result["indicators"]
        +
        attachment_result["findings"]
        +
        header_result["findings"]
    )

    return {
        "ml_score": round(float(ml_score), 2),
        "rule_score": total_rule_score,
        "final_score": round(float(final_score), 2),
        "risk_level": risk_level,
        "indicators": indicators
    }

In [64]:
email = """
URGENT

Your Microsoft account has been suspended.

Verify immediately.

http://192.168.1.1/login

https://bit.ly/reset123
"""

result = analyze_email_complete(
    email_text=email,
    attachments=[
        "invoice.pdf.exe",
        "resume.docm"
    ],
    from_email="support@microsoft.com",
    reply_to="fake@gmail.com"
)

print(result)

{'ml_score': 69.74, 'rule_score': 90, 'final_score': 75.82, 'risk_level': 'Critical', 'indicators': ["Urgent language detected (['urgent', 'verify'])", 'IP-based URL detected', 'Shortened URL detected', 'Dangerous attachment: invoice.pdf.exe', 'Dangerous attachment: resume.docm', 'Reply-To mismatch detected']}


In [65]:
!pip install streamlit pyngrok

In [68]:
%%writefile app.py

Overwriting app.py


In [69]:
%%writefile app.py

import streamlit as st

st.set_page_config(
    page_title="AI Phishing Email Analyzer",
    layout="wide"
)

st.title("AI-Powered Phishing Email Analyzer")

email_text = st.text_area(
    "Paste Email Content",
    height=250
)

st.write("Hello Streamlit")

Overwriting app.py


In [70]:
!ls

20030228_easy_ham.tar.bz2  easy_ham		 report_generator.py
20030228_spam.tar.bz2	   feature_extractor.py  risk_engine.py
app.py			   models		 rule_engine.py
classifier.py		   parser.py		 spam
dashboard.py		   __pycache__		 spamassassin_processed.csv


In [71]:
!pip install -q streamlit pyngrok

In [72]:
%%writefile app.py

import streamlit as st

st.set_page_config(
    page_title="AI Phishing Email Analyzer",
    layout="wide"
)

st.title(" AI-Powered Phishing Email Analyzer")

st.success("Streamlit is working successfully!")

Overwriting app.py


In [73]:
!streamlit run app.py &>/content/logs.txt &

In [75]:
from google.colab import files

files.download("models/model.pkl")
files.download("models/vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [76]:
import zipfile

files_to_zip = [
    "parser.py",
    "feature_extractor.py",
    "rule_engine.py",
    "risk_engine.py",
    "report_generator.py",
    "dashboard.py",
    "classifier.py",
    "app.py"
]

with zipfile.ZipFile("phishing_analyzer_project.zip", "w") as zipf:
    for file in files_to_zip:
        zipf.write(file)

print("ZIP file created successfully!")

ZIP file created successfully!


In [77]:
from google.colab import files

files.download("phishing_analyzer_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [78]:
import zipfile
import os

with zipfile.ZipFile("phishing_analyzer_complete.zip", "w") as zipf:

    project_files = [
        "parser.py",
        "feature_extractor.py",
        "rule_engine.py",
        "risk_engine.py",
        "report_generator.py",
        "dashboard.py",
        "classifier.py",
        "app.py"
    ]

    for file in project_files:
        if os.path.exists(file):
            zipf.write(file)

    if os.path.exists("models/model.pkl"):
        zipf.write("models/model.pkl")

    if os.path.exists("models/vectorizer.pkl"):
        zipf.write("models/vectorizer.pkl")

print("Complete project ZIP created!")

Complete project ZIP created!


In [79]:
for file in [
    "parser.py",
    "feature_extractor.py",
    "rule_engine.py",
    "risk_engine.py",
    "report_generator.py",
    "dashboard.py",
    "classifier.py"
]:
    print(f"\n--- {file} ---")
    !wc -l $file


--- parser.py ---
0 parser.py

--- feature_extractor.py ---
0 feature_extractor.py

--- rule_engine.py ---
0 rule_engine.py

--- risk_engine.py ---
0 risk_engine.py

--- report_generator.py ---
0 report_generator.py

--- dashboard.py ---
0 dashboard.py

--- classifier.py ---
0 classifier.py


In [80]:
%%writefile analysis_engine.py

Writing analysis_engine.py


In [81]:
# Imports

import re
import joblib
import numpy as np

# Load Model

model = joblib.load("models/model.pkl")
vectorizer = joblib.load("models/vectorizer.pkl")

# ------------------
# Parser
# ------------------

URL_REGEX = r'https?://[^\s]+'

def extract_urls(text):
    return re.findall(URL_REGEX, text)

def parse_email_text(email_text):

    return {
        "body": email_text,
        "urls": extract_urls(email_text)
    }

# ------------------
# Feature Extraction
# ------------------

URGENT_WORDS = [
    "urgent",
    "verify",
    "account suspended",
    "click now",
    "immediate action",
    "password reset",
    "security alert"
]

SHORTENERS = [
    "bit.ly",
    "tinyurl.com",
    "goo.gl",
    "t.co"
]

def detect_urgent_words(text):

    text = text.lower()

    return [
        word
        for word in URGENT_WORDS
        if word in text
    ]

def capital_ratio(text):

    letters = [
        c
        for c in text
        if c.isalpha()
    ]

    if not letters:
        return 0

    capitals = [
        c
        for c in letters
        if c.isupper()
    ]

    return len(capitals)/len(letters)

def is_ip_url(url):

    return bool(
        re.search(
            r'https?://\d+\.\d+\.\d+\.\d+',
            url
        )
    )

def is_shortened_url(url):

    return any(
        s in url
        for s in SHORTENERS
    )

def extract_features(parsed_email):

    urls = parsed_email["urls"]
    body = parsed_email["body"]

    return {
        "url_count": len(urls),
        "urgent_words": detect_urgent_words(body),
        "urgent_count": len(
            detect_urgent_words(body)
        ),
        "capital_ratio": capital_ratio(body),
        "ip_urls": sum(
            is_ip_url(url)
            for url in urls
        ),
        "short_urls": sum(
            is_shortened_url(url)
            for url in urls
        )
    }

In [82]:
!wc -l analysis_engine.py

0 analysis_engine.py


In [86]:
%%writefile analysis_engine.py

import re
import joblib
import numpy as np

# ======================
# Load Model
# ======================

model = joblib.load("models/model.pkl")
vectorizer = joblib.load("models/vectorizer.pkl")

# ======================
# Text Preprocessing
# ======================

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\\S+", "", text)
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    return text

# ======================
# Email Parser
# ======================

URL_REGEX = r'https?://[^\\s]+'

def extract_urls(text):
    return re.findall(URL_REGEX, text)

def parse_email_text(email_text):

    return {
        "body": email_text,
        "urls": extract_urls(email_text)
    }

# ======================
# Feature Extraction
# ======================

URGENT_WORDS = [
    "urgent",
    "verify",
    "account suspended",
    "click now",
    "immediate action",
    "password reset",
    "security alert"
]

SHORTENERS = [
    "bit.ly",
    "tinyurl.com",
    "goo.gl",
    "t.co"
]

def detect_urgent_words(text):

    text = text.lower()

    return [
        word
        for word in URGENT_WORDS
        if word in text
    ]

def capital_ratio(text):

    letters = [
        c
        for c in text
        if c.isalpha()
    ]

    if len(letters) == 0:
        return 0

    capitals = [
        c
        for c in letters
        if c.isupper()
    ]

    return len(capitals) / len(letters)

def is_ip_url(url):

    return bool(
        re.search(
            r'https?://\\d+\\.\\d+\\.\\d+\\.\\d+',
            url
        )
    )

def is_shortened_url(url):

    return any(
        s in url
        for s in SHORTENERS
    )

def extract_features(parsed_email):

    urls = parsed_email["urls"]
    body = parsed_email["body"]

    return {
        "url_count": len(urls),
        "urgent_words": detect_urgent_words(body),
        "urgent_count": len(
            detect_urgent_words(body)
        ),
        "capital_ratio": capital_ratio(body),
        "ip_urls": sum(
            is_ip_url(url)
            for url in urls
        ),
        "short_urls": sum(
            is_shortened_url(url)
            for url in urls
        )
    }

Overwriting analysis_engine.py


In [87]:
!wc -l analysis_engine.py

125 analysis_engine.py


In [85]:
!wc -l analysis_engine.py

3 analysis_engine.py


In [88]:
%%writefile -a analysis_engine.py

# ======================
# Rule Engine
# ======================

def calculate_rule_score(features):

    score = 0
    indicators = []

    if features["urgent_count"] > 0:
        score += 10
        indicators.append(
            f"Urgent language detected ({features['urgent_words']})"
        )

    if features["url_count"] > 3:
        score += 10
        indicators.append(
            "Multiple URLs detected"
        )

    if features["ip_urls"] > 0:
        score += 15
        indicators.append(
            "IP-based URL detected"
        )

    if features["short_urls"] > 0:
        score += 10
        indicators.append(
            "Shortened URL detected"
        )

    if features["capital_ratio"] > 0.30:
        score += 5
        indicators.append(
            "Excessive capitalization"
        )

    return {
        "rule_score": min(score, 100),
        "indicators": indicators
    }

# ======================
# Attachment Analysis
# ======================

DANGEROUS_EXTENSIONS = [
    ".exe",
    ".bat",
    ".js",
    ".vbs",
    ".scr",
    ".ps1",
    ".docm",
    ".xlsm"
]

def analyze_attachments(files):

    findings = []
    score = 0

    for file in files:

        file = file.lower()

        for ext in DANGEROUS_EXTENSIONS:

            if file.endswith(ext):

                findings.append(
                    f"Dangerous attachment: {file}"
                )

                score += 20

    return {
        "attachment_score": score,
        "findings": findings
    }

# ======================
# Header Analysis
# ======================

def analyze_headers(
    from_email,
    reply_to
):

    findings = []
    score = 0

    if (
        from_email
        and reply_to
        and from_email.lower()
        != reply_to.lower()
    ):

        findings.append(
            "Reply-To mismatch detected"
        )

        score += 15

    return {
        "header_score": score,
        "findings": findings
    }

# ======================
# ML Scoring
# ======================

def get_ml_score(email_text):

    clean_text = preprocess(email_text)

    vector = vectorizer.transform(
        [clean_text]
    )

    probability = model.predict_proba(
        vector
    )[0][1]

    return float(probability * 100)

# ======================
# Risk Fusion
# ======================

def calculate_final_risk(
    ml_score,
    rule_score
):

    return round(
        (0.7 * ml_score)
        +
        (0.3 * rule_score),
        2
    )

def classify_risk(score):

    if score < 25:
        return "Low"

    elif score < 50:
        return "Medium"

    elif score < 75:
        return "High"

    return "Critical"

Appending to analysis_engine.py


In [89]:
!wc -l analysis_engine.py

285 analysis_engine.py


In [90]:
import analysis_engine

print("Imported Successfully")

Imported Successfully


In [91]:
%%writefile -a analysis_engine.py

# ======================
# OpenRouter Integration
# ======================

import os
from openai import OpenAI

def get_openrouter_client():

    api_key = os.getenv("OPENROUTER_API_KEY")

    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )

# ======================
# Prompt Builder
# ======================

def build_prompt(
    email_text,
    analysis_result
):

    return f"""
You are a cybersecurity phishing analyst.

EMAIL:
{email_text}

RISK SCORE:
{analysis_result['final_score']}

RISK LEVEL:
{analysis_result['risk_level']}

INDICATORS:
{analysis_result['indicators']}

Provide:

1. Threat Summary

2. Risk Explanation

3. Attack Category
(Credential Theft, Malware, BEC, Scam, Other)

4. Recommendations

5. Security Awareness Tips

Keep the response concise and professional.
"""

# ======================
# Threat Report
# ======================

def generate_threat_report(
    email_text,
    analysis_result,
    model_name="openai/gpt-4o-mini"
):

    client = get_openrouter_client()

    prompt = build_prompt(
        email_text,
        analysis_result
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# ======================
# Complete Analysis
# ======================

def analyze_email_complete(
    email_text,
    attachments=None,
    from_email=None,
    reply_to=None
):

    if attachments is None:
        attachments = []

    parsed = parse_email_text(
        email_text
    )

    features = extract_features(
        parsed
    )

    rule_result = calculate_rule_score(
        features
    )

    ml_score = get_ml_score(
        email_text
    )

    attachment_result = analyze_attachments(
        attachments
    )

    header_result = analyze_headers(
        from_email,
        reply_to
    )

    total_rule_score = (
        rule_result["rule_score"]
        +
        attachment_result["attachment_score"]
        +
        header_result["header_score"]
    )

    total_rule_score = min(
        total_rule_score,
        100
    )

    final_score = calculate_final_risk(
        ml_score,
        total_rule_score
    )

    risk_level = classify_risk(
        final_score
    )

    indicators = (
        rule_result["indicators"]
        +
        attachment_result["findings"]
        +
        header_result["findings"]
    )

    return {
        "ml_score": round(
            float(ml_score), 2
        ),
        "rule_score": total_rule_score,
        "final_score": round(
            float(final_score), 2
        ),
        "risk_level": risk_level,
        "indicators": indicators
    }

Appending to analysis_engine.py


In [92]:
!wc -l analysis_engine.py

452 analysis_engine.py


In [94]:
!grep -n "def analyze_email_complete" analysis_engine.py

377:def analyze_email_complete(


In [95]:
import importlib
import analysis_engine

importlib.reload(analysis_engine)

print("Reloaded successfully")

Reloaded successfully


In [96]:
print(hasattr(analysis_engine, "analyze_email_complete"))

True


In [97]:
[x for x in dir(analysis_engine) if not x.startswith("_")]

['DANGEROUS_EXTENSIONS',
 'OpenAI',
 'SHORTENERS',
 'URGENT_WORDS',
 'URL_REGEX',
 'analyze_attachments',
 'analyze_email_complete',
 'analyze_headers',
 'build_prompt',
 'calculate_final_risk',
 'calculate_rule_score',
 'capital_ratio',
 'classify_risk',
 'detect_urgent_words',
 'extract_features',
 'extract_urls',
 'generate_threat_report',
 'get_ml_score',
 'get_openrouter_client',
 'is_ip_url',
 'is_shortened_url',
 'joblib',
 'model',
 'np',
 'os',
 'parse_email_text',
 'preprocess',
 're',
 'vectorizer']

In [98]:
from google.colab import files

files.download("analysis_engine.py")
files.download("models/model.pkl")
files.download("models/vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>